# 🧪 W5-D7 概念实验：技术路由、Pareto 选型、四层防线与生成-验证-重试

> 配套阅读：`ima/第5周-Day7-第五周总复习.md`（本周技术地图、能力链、决策树在那边）
>
> 本 notebook 回答四个问题：
> 1. "什么问题用什么推理技术"能否写成**可执行的路由规则**？
> 2. 在成本-准确率平面上，如何用 Pareto 前沿做选型？
> 3. 四层防线（Prompt/推理/证据/治理）的**乘法效应**有多大？
> 4. 端到端"生成→验证→重试"为什么能把 70% 拉到 96%+？前提是什么？

## 实验 1：把"技术选择决策树"写成可执行路由器

决策树三个信号：是否需要外部数据（库存/订单/实时）、是否多步推理、是否高风险。
规则版本就是 md 里决策树的直译——能跑、能测、能进版本控制。

In [ ]:
def route(question):
    needs_external = any(k in question for k in ["今天", "库存", "订单", "余额", "最新", "报价", "到哪"])
    multi_step = any(k in question for k in ["计算", "排班", "预算", "排查", "规划", "对账", "算"])
    high_stakes = any(k in question for k in ["退款", "合同", "财务", "医疗", "法律", "调薪"])
    if needs_external and multi_step:
        return "Agent：先取数再分步推理，关键节点人工复核"
    if needs_external:
        return "工具调用 + 简短回答（不需要推理链）"
    if multi_step and high_stakes:
        return "CoT + Self-consistency 多链投票 + 人工抽检"
    if multi_step:
        return "CoT 单链（够用，别浪费 Token）"
    return "Zero-shot 直接回答"

tests = [
    "今天上海天气怎样？",
    "帮我算这个月的毛利率",
    "算一下全员调薪后的季度预算",
    "订单 1024 现在到哪了？",
    "这笔退款该不该批？需要结合政策算金额",
    "把这段售后话术翻译成英文",
]
for t in tests:
    print(f"「{t}」\n  → {route(t)}")

## 实验 2：成本-准确率平面上的 Pareto 选型

五种技术各在 300 道模拟题上评测（准确率带随机波动）。
Pareto 前沿上的点 = 没有别的技术"更便宜又更准"。被支配的技术只有在特定预算/约束下才考虑。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

rng = np.random.default_rng(3)
n_q = 300
techs = {   # (真实准确率, 相对成本)
    "Zero-shot":     (0.55, 1.0),
    "Few-shot":      (0.63, 2.2),
    "CoT":           (0.71, 4.5),
    "CoT+验证重试":  (0.86, 6.0),
    "CoT+自洽(8链)": (0.89, 30.0),
}

pts = []
for name, (p, cost) in techs.items():
    acc = (rng.random(n_q) < p).mean()          # 一次评测的抽样波动
    pts.append((cost, acc, name))

front = sorted([p for p in pts if not any(q[0] <= p[0] and q[1] >= p[1] and q is not p for q in pts)])
front_names = {f[2] for f in front}

plt.figure(figsize=(7, 4.2))
for cost, acc, name in pts:
    plt.scatter(cost, acc * 100, s=90,
                color="#fb8500" if name in front_names else "#adb5bd")
    plt.annotate(name, (cost, acc * 100), fontsize=9, xytext=(6, 4), textcoords="offset points")
plt.plot([f[0] for f in front], [f[1] * 100 for f in front], "r--", lw=1, alpha=0.6)
plt.xscale("log")
plt.xlabel("单题相对成本（log）"); plt.ylabel("评测准确率 (%)")
plt.title("技术选型：橙点=Pareto 前沿；灰点被支配")
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Pareto 前沿:", " → ".join(f[2] for f in front))
print("注意：某次评测的抽样波动可能临时改变前沿成员 → 选型也要看置信区间（见 Day4）。")

## 实验 3：四层防线的乘法效应

初始 20% 的请求"会出错"。四层防线各拦截一部分：
Prompt 约束 50% → 分步推理+自查 40% → 工具/数据验证 60% → 治理与人工抽检 30%。
错误残留率是**乘法**衰减：$0.20 \times 0.50 \times 0.60 \times 0.40 \times 0.70$。
对比：把全部预算堆在一层"拦截率 90% 的超级 Prompt"上。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

rng = np.random.default_rng(1)
n = 100_000
layers = [("Prompt 约束", 0.50), ("分步推理+自查", 0.40),
          ("工具/数据验证", 0.60), ("治理与人工抽检", 0.30)]

e = rng.random(n) < 0.20
residual, names = [e.mean()], ["无防线"]
for name, c in layers:
    e = e & (rng.random(n) > c)          # 该层拦截 c 比例的错误
    residual.append(e.mean()); names.append("+" + name)

plt.figure(figsize=(8, 4))
plt.bar(range(len(names)), [r * 100 for r in residual], color="#219ebc")
plt.xticks(range(len(names)), names, rotation=15, fontsize=8)
plt.ylabel("错误残留率 (%)")
plt.title("四层防线：每层只拦一部分，乘起来效果惊人")
for i, r in enumerate(residual):
    plt.text(i, r * 100 + 0.15, f"{r:.2%}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

four_layer = 0.20 * float(np.prod([1 - c for _, c in layers]))
one_layer = 0.20 * 0.10
print(f"四层各拦 40~60% → 残留 {four_layer:.2%}")
print(f"单层拦截 90%（超级 Prompt）→ 残留 {one_layer:.2%}")
print("四层没有任何一层做到 90%，残留率反而更低——而且每层防的失效模式互不相同：")
print("Prompt 再强也验证不了'引用的库存数字是不是真的'，那要靠证据层。纵深防御赢在乘法。")

## 实验 4：端到端"生成→验证→重试"——可验证奖励的威力

单次生成正确率只有 70%，但配一个能抓住 90% 错误的**验证器**（重算/单测/规则）：
错了且被抓住 → 重试；错了且漏检 → 泄漏。统计重试 0~3 次的最终正确率与平均生成次数。
这正是 R1 系 RL（可验证奖励）与 Agent self-check 共同的根基。

In [ ]:
import numpy as np

rng = np.random.default_rng(6)
n_q = 100_000
p_gen, p_catch = 0.70, 0.90

def pipeline(max_retries):
    correct = rng.random(n_q) < p_gen
    uncaught = ~correct & (rng.random(n_q) > p_catch)     # 错了但验证器没抓住
    attempts = np.ones(n_q)
    for _ in range(max_retries):
        retry = ~correct & ~uncaught                      # 错了且被抓住 → 重试
        if not retry.any():
            break
        idx = np.where(retry)[0]
        again = rng.random(len(idx)) < p_gen
        again_caught = ~again & (rng.random(len(idx)) < p_catch)
        correct[idx] = again
        uncaught[idx] = ~again & ~again_caught
        attempts[idx] += 1
    return correct.mean(), attempts.mean()

for mr in (0, 1, 2, 3):
    acc, avg = pipeline(mr)
    print(f"重试上限 {mr} 次 → 最终正确率 {acc:7.3%} | 平均生成次数 {avg:.2f}")
print()
print("70% 的生成器 + 90% 的验证器 + 3 次重试 ≈ 96%+ 的系统。")
print("前提只有一个：验证器真的能判对错。验证器若只能抓 50% 的错，收益立刻腰斩——")
print("这就是'可验证奖励'（数学/代码/规则）是推理能力根基的原因。")

## 结论

- 技术选型可以（也应该）从散文变成**可执行路由规则**，进版本控制、可回归测试（实验 1）
- Pareto 前沿 + 预算线 = 选型方法；评测波动会影响前沿成员，要看区间（实验 2）
- 可靠性来自纵深：各层失效模式独立，错误残留乘法衰减（实验 3）
- "会验证"解锁"可重试"：70% 生成器 + 90% 验证器 → 96%+ 系统（实验 4）

→ 深入阅读：`ima/第5周-Day7-第五周总复习.md`（W1→W5 能力链、本周技术地图、一页纸复盘法）